# IEEE-CIS Fraud Detection - Model Inference

## Setup

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import dagshub

dagshub.init(
    repo_owner='sansi23',
    repo_name='IEEE-CIS-Fraud-Detection',
    mlflow=True
)

Accessing as sansi23

Initialized MLflow to track repo "sansi23/IEEE-CIS-Fraud-Detection"

Repository sansi23/IEEE-CIS-Fraud-Detection initialized!

## Load Best Model

In [11]:
model_name    = 'XGBoost_FraudDetection_Pipeline'
model_version = 14

model_uri = f'models:/{model_name}/{model_version}'
pipeline  = mlflow.sklearn.load_model(model_uri)

print('Model:', type(pipeline))
print('Steps:', [name for name, _ in pipeline.steps])

Model: <class 'sklearn.pipeline.Pipeline'>
Steps: ['preprocessor', 'corr_dropper', 'iv_selector', 'scaler', 'classifier']


## Load Test Data


In [12]:
test_transaction = pd.read_csv('test_transaction.csv')
test_identity    = pd.read_csv('test_identity.csv')

test     = test_transaction.merge(test_identity, on='TransactionID', how='left')
test_ids = test['TransactionID']
X_test   = test.drop(columns=['TransactionID'])

print('X_test shape:', X_test.shape)

X_test shape: (506691, 432)


## Generate Predictions & Submission

In [13]:
y_proba = pipeline.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'TransactionID': test_ids,
    'isFraud'      : y_proba
})

submission.to_csv('submission.csv', index=False)

print(submission.shape)
print(submission.head())

(506691, 2)
   TransactionID   isFraud
0        3663549  0.164658
1        3663550  0.320739
2        3663551  0.250514
3        3663552  0.107345
4        3663553  0.267069
